## 1 Functions and module

In [1]:
import pandas as pd
import gzip
import glob
import os
import subprocess

In [3]:
nb = globals().get("__vsc_ipynb_file__")
if isinstance(nb, str) and nb:
    try:
        os.chdir(os.path.dirname(nb))
    except Exception:
        pass  # silently ignore any failure
# else: do nothing outside VS Code


In [4]:
def get_ngs_dir(config_path,variable):
    """
    Returns the value of NGS_DIR defined in the given config.sh file,
    or None if it's not found or there's an error.
    """
    # Run a subshell that sources config.sh and echoes $NGS_DIR
    command = f"bash -c 'source {config_path} && echo ${variable}'"
    try:
        result = subprocess.run(
            command, shell=True, check=True, capture_output=True, text=True
        )
        ngs_dir = result.stdout.strip()
        # If NGS_DIR is not set or empty, return None
        return ngs_dir if ngs_dir else None
    except subprocess.CalledProcessError:
        return None


## 2 Input and output address

In [5]:
parental_address = '../../'

## 3 Load config.sh

In [6]:
temp = 'NGS_DIR'
ngs_dir = get_ngs_dir(f"{parental_address}/config.sh",temp)
print(f"{temp}= {ngs_dir}")


NGS_DIR= /labs/mwinslow/Haiqing/NGS_Raw_data/LA103_L1_251010


In [7]:
temp = 'PROJECT_DIR'
project_dir = get_ngs_dir(f"{parental_address}/config.sh",temp)
print(f"{temp}= {project_dir}")


PROJECT_DIR= /labs/mwinslow/Haiqing/UltraSeq_Projects/EGFR_Lkb1_effector_tumor


In [13]:
fastq_dir = ngs_dir+"/01.RawData"
print(fastq_dir)

output_prefix = project_dir+"/01_data_collection/data"
print(output_prefix1)

/labs/mwinslow/Haiqing/NGS_Raw_data/LA103_L1_251010/01.RawData
/labs/mwinslow/Haiqing/UltraSeq_Projects/EGFR_Lkb1_effector_tumor/01_data_collection/data


## 4 Generate file with necessary raw data address

In [9]:
# find all the fastq.gz files 
temp_pattern = '/**/*fq.gz' # When recursive is set, ** followed by a path separator matches 0 or more subdirectories.
fastq_address_list = glob.glob(fastq_dir+temp_pattern, recursive=True)

In [ ]:
fastq_address_list

In [11]:
# Initialize an empty dictionary to organize FASTQ files
# Structure: { sample_id: { sub_id: [list of file paths] } }
sample_dic = {}
# Iterate over all FASTQ file paths in sorted order
for file_path in sorted(fastq_address_list):
    
    # Extract sample ID from the second-to-last element in the path
    # Example: "/.../sample123/sub001_R1.fastq.gz" → "sample123"
    sample_id = file_path.split('/')[-2]
    
    # Extract sub-ID from the filename by removing the last 8 characters
    # This assumes the suffix is always "_R1.fastq.gz" or similar
    # Example: "sub001_R1.fastq.gz"[:-8] → "sub001"
    sub_id = file_path.split('/')[-1][:-8]
    
    # If this is the first time seeing this sample ID, create a new entry
    if sample_id not in sample_dic:
        sample_dic[sample_id] = {sub_id: [file_path]}
    
    # If the sample ID exists but the sub-ID is new, initialize a new list
    elif sub_id not in sample_dic[sample_id]:
        sample_dic[sample_id][sub_id] = [file_path]
    
    # If both sample ID and sub-ID exist, append the file path
    else:
        sample_dic[sample_id][sub_id].append(file_path)


In [14]:
# Ensure the output directory exists
raw_reads_dir = os.path.join(output_prefix, "Raw_reads")
os.makedirs(raw_reads_dir, exist_ok=True)

sample_id_list = []
address1_list = []
address2_list = []

for sample_id, subdict in sample_dic.items():
    sample_id_list.append(sample_id)
    
    # Process read 1
    fastq_list_r1 = sorted([x[0] for x in subdict.values()])
    merged_file_r1 = os.path.join(raw_reads_dir, f"{sample_id}_combined_1.fq.gz")
    address1_list.append(merged_file_r1)
    
    # Merge the read 1 files using cat.
    # Note: This produces a multi-member gzip file, which is acceptable for most tools.
    cmd_r1 = f"cat {' '.join(fastq_list_r1)} > {merged_file_r1}"
    subprocess.run(cmd_r1, shell=True, check=True)

    # Process read 2
    fastq_list_r2 = sorted([x[1] for x in subdict.values()])
    merged_file_r2 = os.path.join(raw_reads_dir, f"{sample_id}_combined_2.fq.gz")
    address2_list.append(merged_file_r2)
    
    # Merge the read 2 files using cat.
    cmd_r2 = f"cat {' '.join(fastq_list_r2)} > {merged_file_r2}"
    subprocess.run(cmd_r2, shell=True, check=True)

    print(f"Merged files for sample: {sample_id}")

Merged files for sample: LA103_03
Merged files for sample: LA103_05
Merged files for sample: LA103_06
Merged files for sample: LA103_07
Merged files for sample: LA103_08
Merged files for sample: LA103_09
Merged files for sample: LA103_11
Merged files for sample: LA103_12
Merged files for sample: LA103_13
Merged files for sample: LA103_14
Merged files for sample: LA103_15
Merged files for sample: LA103_16
Merged files for sample: LA103_17
Merged files for sample: LA103_18
Merged files for sample: LA103_19
Merged files for sample: LA103_20
Merged files for sample: LA103_21
Merged files for sample: LA103_22
Merged files for sample: LA103_23
Merged files for sample: LA103_24
Merged files for sample: LA103_25
Merged files for sample: LA103_26
Merged files for sample: LA103_27
Merged files for sample: LA103_28
Merged files for sample: LA103_29
Merged files for sample: LA103_30
Merged files for sample: LA103_32
Merged files for sample: LA103_33
Merged files for sample: LA103_34
Merged files f

In [15]:
read_df = pd.DataFrame({'Sample_ID':sample_id_list,
                        'Address_r1':address1_list,
                        'Address_r2':address2_list,})

In [16]:
read_df.shape

(68, 3)

In [17]:
read_df.head()

,Sample_ID,Address_r1,Address_r2
0,LA103_03,/labs/mwinslow/Haiqing/UltraSeq_Projects/EGFR_...,/labs/mwinslow/Haiqing/UltraSeq_Projects/EGFR_...
1,LA103_05,/labs/mwinslow/Haiqing/UltraSeq_Projects/EGFR_...,/labs/mwinslow/Haiqing/UltraSeq_Projects/EGFR_...
2,LA103_06,/labs/mwinslow/Haiqing/UltraSeq_Projects/EGFR_...,/labs/mwinslow/Haiqing/UltraSeq_Projects/EGFR_...
3,LA103_07,/labs/mwinslow/Haiqing/UltraSeq_Projects/EGFR_...,/labs/mwinslow/Haiqing/UltraSeq_Projects/EGFR_...
4,LA103_08,/labs/mwinslow/Haiqing/UltraSeq_Projects/EGFR_...,/labs/mwinslow/Haiqing/UltraSeq_Projects/EGFR_...


In [19]:
print(f'There are {read_df.shape[0]} samples for the this project')

There are 68 samples for the Lkb1-effector project


## 5 Output data

In [20]:
temp_o = output_prefix1 +"/NGS_address"

In [21]:
file_a = open(temp_o, 'w')
for index, row in read_df.iterrows():
    t1 = row['Address_r1']
    t2 = row['Address_r2']
    t3 = row['Sample_ID']
    temp_s = ','.join([t1,t2,t3])+'\n'
    file_a.write(temp_s)
file_a.close()